# 01 - Eddy covariance flux processing

Each zip file from the flux tower holds one half-hour of EddyPro (SMARTFlux) output.
This notebook:

1. stacks the half-hourly `eddypro_exp_full_output` rows for one tower,
2. builds the input table for the Kljun et al. (2015) FFP footprint model (used in notebook 04),
3. aggregates half-hourly ET to daily ETa, which is the target variable for the ML models.

Run it once for each site (`ASP` = EC-1, `BAU` = EC-2).

In [ ]:
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd

In [ ]:
SITE = "ASP"   # "ASP" (EC-1) or "BAU" (EC-2)

# expected layout: data/raw/eddy_covariance/<SITE>/<YYYY>/<MM>/*.zip
RAW_DIR = Path("../data/raw/eddy_covariance") / SITE
OUT_DIR = Path("../data/processed/ec")
OUT_DIR.mkdir(parents=True, exist_ok=True)

## Read the half-hourly EddyPro output

In the full output csv, row 2 is the header and row 3 usually holds the units, followed by the data row.
Some files have no units row, so the units row is detected by its content rather than assumed.
Corrupted zips are skipped and reported.

In [ ]:
def is_year_dir(p):
    return p.is_dir() and p.name.isdigit() and len(p.name) == 4


def is_month_dir(p):
    return p.is_dir() and p.name.isdigit() and len(p.name) == 2 and 1 <= int(p.name) <= 12


def read_eddypro_csv(z, inner_csv):
    with z.open(inner_csv) as fh:
        df = pd.read_csv(fh, header=1, nrows=2, engine="python", on_bad_lines="skip")
    # drop the units row ("[yyyy-mm-dd]", "[HH:MM]", ...) when the file has one
    df = df[~df["date"].astype(str).str.startswith("[")]
    return df.head(1)


rows = []
for year_dir in sorted(p for p in RAW_DIR.iterdir() if is_year_dir(p)):
    for month_dir in sorted(p for p in year_dir.iterdir() if is_month_dir(p)):
        for zip_path in month_dir.glob("*.zip"):
            try:
                with zipfile.ZipFile(zip_path, "r") as z:
                    csvs = [f for f in z.namelist()
                            if f.startswith("output/eddypro_exp_full_output") and f.endswith(".csv")]
                    for inner_csv in csvs:
                        df = read_eddypro_csv(z, inner_csv)
                        df.insert(0, "year", int(year_dir.name))
                        df.insert(1, "month", int(month_dir.name))
                        df.insert(2, "source_zip", zip_path.name)
                        rows.append(df)
            except Exception as e:
                print(f"skipped {zip_path.name}: {e}")

all_data = pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()
print(f"{len(all_data)} half-hourly records read for {SITE}")
all_data.head()

## Inputs for the FFP footprint model

The FFP model needs, for every half hour: measurement height above displacement (zm), displacement
height (d), roughness length (z0), mean wind speed, Obukhov length (L), the standard deviation of
the lateral wind (sigma_v), friction velocity (u*) and wind direction.
Column names are not always the same between EddyPro versions, so a few aliases are checked.
zm is recovered from the stability parameter as zm = L * (z-d)/L.

In [ ]:
def make_ffp_table(all_data):
    df = all_data.copy()

    dt = pd.to_datetime(df["date"], errors="coerce")
    hhmm = df["time"].astype(str).str.split(":", n=1, expand=True)

    if "U_mean" in df.columns:
        U_mean = pd.to_numeric(df["U_mean"], errors="coerce")
    else:
        U_mean = pd.to_numeric(df.get("u_rot", np.nan), errors="coerce")

    L = pd.to_numeric(df["L"], errors="coerce") if "L" in df.columns else np.nan

    # stability parameter column is written as "(z-d)/L"
    zdl_col = next((c for c in df.columns if "z-d" in c and "/L" in c), None)
    zdl = pd.to_numeric(df[zdl_col], errors="coerce") if zdl_col else np.nan
    zm = L * zdl

    d = pd.to_numeric(df.get("d", np.nan), errors="coerce")
    z0 = pd.to_numeric(df.get("z0", np.nan), errors="coerce")

    # sigma_v from the variance of the v wind component
    vvar_col = None
    for c in df.columns:
        if c.lower().replace("_", "") in ("vvar", "vvariance") or "v_var" in c.lower():
            vvar_col = c
            break
    sigma_v = np.sqrt(pd.to_numeric(df[vvar_col], errors="coerce")) if vvar_col else np.nan

    u_star = np.nan
    for c in ("u*", "u_star", "u-star"):
        if c in df.columns:
            u_star = pd.to_numeric(df[c], errors="coerce")
            break

    if "wind_dir" in df.columns:
        wind_dir = pd.to_numeric(df["wind_dir"], errors="coerce")
    else:
        alt = next((c for c in df.columns if "wind" in c.lower() and "dir" in c.lower()), None)
        wind_dir = pd.to_numeric(df[alt], errors="coerce") if alt else np.nan

    return pd.DataFrame({
        "yyyy": dt.dt.year,
        "mm": dt.dt.month,
        "day": dt.dt.day,
        "HU_UTC": pd.to_numeric(hhmm[0], errors="coerce"),
        "MM": pd.to_numeric(hhmm[1], errors="coerce"),
        "zm": zm,
        "d": d,
        "z0": z0,
        "U_mean": U_mean,
        "L": L,
        "sigma_v": sigma_v,
        "u_star": u_star,
        "wind_dir": wind_dir,
    })


ffp_df = make_ffp_table(all_data)
ffp_df.to_csv(OUT_DIR / f"{SITE}_ffp_inputs.csv", index=False)
ffp_df.head()

## Daily ETa

EddyPro reports ET for each half hour as a rate in mm/h, so the daily total is the sum of the
half-hourly rates divided by 2. Negative half-hourly values are set to zero before summing.

In [ ]:
et = pd.to_numeric(all_data["ET"], errors="coerce").clip(lower=0)
day = pd.to_datetime(all_data["date"].astype(str) + " " + all_data["time"].astype(str),
                     errors="coerce").dt.date

daily_eta = (
    pd.DataFrame({"date": day, "ET": et})
    .groupby("date", as_index=False)["ET"].sum()
)
daily_eta["ETa"] = daily_eta["ET"] / 2.0     # sum of mm/h over 30-min steps -> mm/day
daily_eta = daily_eta[["date", "ETa"]]

daily_eta.to_csv(OUT_DIR / f"{SITE}_daily_eta.csv", index=False)
print(daily_eta.describe())